# Train-Test Split: Efek `random_state` dan `stratify`

Notebook ini memakai data sintetis **10 baris** untuk kasus **klasifikasi multi-kelas**
(penilaian risiko kredit: `Low` / `Medium` / `High`) — supaya setiap baris masih bisa
dilihat satu per satu di slide.

Yang ditunjukkan:

| Bagian | Pertanyaan yang dijawab |
|---|---|
| 1. Data sintetis | Data apa yang dipakai |
| 2. Split dasar | Bagaimana `train_test_split` membagi data |
| 3. `random_state` | Kenapa hasil split bisa berubah-ubah setiap dijalankan |
| 4. `stratify` | Kenapa proporsi kelas di test set bisa melenceng |


## 1. Data sintetis (10 baris, target 3 kelas)

Kasus: memprediksi **tingkat risiko kredit** pemohon pinjaman.

| Kolom | Tipe | Keterangan |
|---|---|---|
| `age` | numerik | usia pemohon (tahun) |
| `annual_income` | numerik | pendapatan per tahun (ribu USD) |
| `credit_score` | numerik | skor kredit (300-850) |
| `loan_amount` | numerik | jumlah pinjaman diajukan (ribu USD) |
| `employment_years` | numerik | lama bekerja (tahun) |
| `education` | kategorikal | High School / Bachelor / Master |
| `owns_house` | kategorikal | Yes / No |
| `risk_level` | **target (3 kelas)** | Low / Medium / High |

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

CLASSES = ['Low', 'Medium', 'High']

df = pd.DataFrame({
    'applicant_id':     [f'A{i:02d}' for i in range(1, 11)],
    'age':              [22, 35, 41, 28, 53, 30, 47, 24, 38, 60],
    'annual_income':    [32, 68, 92, 41, 120, 63, 105, 36, 81, 135],
    'credit_score':     [590, 705, 755, 610, 790, 680, 770, 580, 745, 800],
    'loan_amount':      [25, 20, 35, 30, 40, 18, 45, 28, 25, 30],
    'employment_years': [1, 8, 14, 2, 25, 6, 20, 1, 12, 30],
    'education':        ['High School', 'Bachelor', 'Master', 'High School', 'Master',
                         'Bachelor', 'Bachelor', 'High School', 'Bachelor', 'Master'],
    'owns_house':       ['No', 'Yes', 'Yes', 'No', 'Yes',
                         'No', 'Yes', 'No', 'Yes', 'Yes'],
    'risk_level':       ['High', 'Medium', 'Low', 'High', 'Low',
                         'Medium', 'Low', 'High', 'Low', 'Low'],
})
df

,applicant_id,age,annual_income,credit_score,loan_amount,employment_years,education,owns_house,risk_level
0,A01,22,32,590,25,1,High School,No,High
1,A02,35,68,705,20,8,Bachelor,Yes,Medium
2,A03,41,92,755,35,14,Master,Yes,Low
3,A04,28,41,610,30,2,High School,No,High
4,A05,53,120,790,40,25,Master,Yes,Low
5,A06,30,63,680,18,6,Bachelor,No,Medium
6,A07,47,105,770,45,20,Bachelor,Yes,Low
7,A08,24,36,580,28,1,High School,No,High
8,A09,38,81,745,25,12,Bachelor,Yes,Low
9,A10,60,135,800,30,30,Master,Yes,Low


In [2]:
# Kelasnya sengaja dibuat tidak seimbang: 5 Low, 3 High, 2 Medium
distribusi = pd.DataFrame({
    'jumlah':   df['risk_level'].value_counts().reindex(CLASSES),
    'proporsi': df['risk_level'].value_counts(normalize=True).reindex(CLASSES).round(2),
})
distribusi

,jumlah,proporsi
risk_level,,
Low,5,0.5
Medium,2,0.2
High,3,0.3


## 2. Split dasar

`train_test_split` memisahkan fitur (`X`) dan target (`y`) menjadi bagian latih dan uji.
Dengan `test_size=0.3` dan 10 baris, kita dapat **7 baris train** dan **3 baris test**.

Catatan: `education` dan `owns_house` masih berupa teks. Untuk melatih model, kolom ini
perlu di-encode dulu — dan encoder-nya di-`fit` **hanya pada data train**, setelah split ini.

In [3]:
FEATURES = ['age', 'annual_income', 'credit_score', 'loan_amount',
            'employment_years', 'education', 'owns_house']

X = df[FEATURES]
y = df['risk_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print('Jumlah data train :', len(X_train))
print('Jumlah data test  :', len(X_test))

Jumlah data train : 7
Jumlah data test  : 3


In [4]:
print('--- TRAIN ---')
display(df.loc[X_train.index])
print('--- TEST ---')
display(df.loc[X_test.index])

--- TRAIN ---


,applicant_id,age,annual_income,credit_score,loan_amount,employment_years,education,owns_house,risk_level
0,A01,22,32,590,25,1,High School,No,High
7,A08,24,36,580,28,1,High School,No,High
2,A03,41,92,755,35,14,Master,Yes,Low
9,A10,60,135,800,30,30,Master,Yes,Low
4,A05,53,120,790,40,25,Master,Yes,Low
3,A04,28,41,610,30,2,High School,No,High
6,A07,47,105,770,45,20,Bachelor,Yes,Low


--- TEST ---


,applicant_id,age,annual_income,credit_score,loan_amount,employment_years,education,owns_house,risk_level
8,A09,38,81,745,25,12,Bachelor,Yes,Low
1,A02,35,68,705,20,8,Bachelor,Yes,Medium
5,A06,30,63,680,18,6,Bachelor,No,Medium


## 3. `random_state`: hasil acak vs hasil yang bisa diulang

`train_test_split` mengacak baris sebelum membagi. Tanpa `random_state`, pengacakan
memakai benih (seed) berbeda setiap kali dijalankan, sehingga isi test set berubah-ubah.

### 3a. Tanpa `random_state` — hasil berbeda tiap run

In [5]:
for percobaan in range(1, 4):
    _, X_test_i, _, y_test_i = train_test_split(X, y, test_size=0.3)   # tanpa random_state
    ids = sorted(df.loc[X_test_i.index, 'applicant_id'])
    print(f'Percobaan {percobaan} -> test set: {ids}  kelas: {sorted(y_test_i)}')

Percobaan 1 -> test set: ['A01', 'A03', 'A09']  kelas: ['High', 'Low', 'Low']
Percobaan 2 -> test set: ['A01', 'A04', 'A10']  kelas: ['High', 'High', 'Low']
Percobaan 3 -> test set: ['A03', 'A05', 'A07']  kelas: ['Low', 'Low', 'Low']


### 3b. Dengan `random_state=42` — hasil selalu sama

In [6]:
for percobaan in range(1, 4):
    _, X_test_i, _, y_test_i = train_test_split(X, y, test_size=0.3, random_state=42)
    ids = sorted(df.loc[X_test_i.index, 'applicant_id'])
    print(f'Percobaan {percobaan} -> test set: {ids}  kelas: {sorted(y_test_i)}')

Percobaan 1 -> test set: ['A02', 'A06', 'A09']  kelas: ['Low', 'Medium', 'Medium']
Percobaan 2 -> test set: ['A02', 'A06', 'A09']  kelas: ['Low', 'Medium', 'Medium']
Percobaan 3 -> test set: ['A02', 'A06', 'A09']  kelas: ['Low', 'Medium', 'Medium']


**Kesimpulan bagian 3**

- Tanpa `random_state`: split berubah setiap run → skor model ikut berubah, hasil sulit dibandingkan
  dan tidak bisa direproduksi orang lain.
- Dengan `random_state`: split terkunci → eksperimen bisa diulang dan dibandingkan secara adil.
- Angka `42` tidak istimewa; yang penting nilainya **tetap**, bukan nilainya berapa.

## 4. `stratify`: menjaga proporsi kelas

Proporsi asli target adalah 50% `Low` : 30% `High` : 20% `Medium`. Idealnya train dan test
punya proporsi yang sama. Tanpa `stratify`, pembagian murni acak — dan pada kasus multi-kelas
risikonya lebih besar, karena **ada 3 kelas yang harus kebagian hanya 3 baris test**.

### 4a. Tanpa `stratify` (`random_state=0`)

In [7]:
X_train_ns, X_test_ns, y_train_ns, y_test_ns = train_test_split(
    X, y, test_size=0.3, random_state=0
)

print('TRAIN:'); print(y_train_ns.value_counts().reindex(CLASSES, fill_value=0).to_string())
print('\nTEST:');  print(y_test_ns.value_counts().reindex(CLASSES, fill_value=0).to_string())
print('\nID di test set:', sorted(df.loc[X_test_ns.index, 'applicant_id']))

TRAIN:
risk_level
Low       2
Medium    2
High      3

TEST:
risk_level
Low       3
Medium    0
High      0

ID di test set: ['A03', 'A05', 'A09']


Test set berisi **3 pemohon `Low` saja** — kelas `Medium` dan `High` hilang sepenuhnya.
Model jadi diuji pada satu kelas saja: precision/recall untuk `Medium` dan `High` tidak bisa
dihitung, dan akurasi test set tidak menggambarkan performa sebenarnya.

### 4b. Dengan `stratify=y` (`random_state=0`)

In [8]:
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

print('TRAIN:'); print(y_train_s.value_counts().reindex(CLASSES, fill_value=0).to_string())
print('\nTEST:');  print(y_test_s.value_counts().reindex(CLASSES, fill_value=0).to_string())
print('\nID di test set:', sorted(df.loc[X_test_s.index, 'applicant_id']))

TRAIN:
risk_level
Low       4
Medium    1
High      2

TEST:
risk_level
Low       1
Medium    1
High      1

ID di test set: ['A01', 'A03', 'A06']


Dengan seed yang sama persis, ketiga kelas kebagian 1 baris di test set.

Dua syarat `stratify` yang perlu diingat:

- setiap kelas minimal punya **2 anggota** (kalau tidak, sklearn error);
- ukuran test set minimal **sebanyak jumlah kelas** (di sini 3 baris untuk 3 kelas).

### 4c. Perbandingan proporsi kelas

In [9]:
def proporsi(series):
    return series.value_counts(normalize=True).reindex(CLASSES).fillna(0)

perbandingan = pd.DataFrame({
    'Original data':       proporsi(y),
    'Train (no stratify)': proporsi(y_train_ns),
    'Test (no stratify)':  proporsi(y_test_ns),
    'Train (stratify)':    proporsi(y_train_s),
    'Test (stratify)':     proporsi(y_test_s),
}).round(2)
perbandingan

,Original data,Train (no stratify),Test (no stratify),Train (stratify),Test (stratify)
risk_level,,,,,
Low,0.5,0.29,1.0,0.57,0.33
Medium,0.2,0.29,0.0,0.14,0.33
High,0.3,0.43,0.0,0.29,0.33


### 4d. Seberapa sering test set kehilangan salah satu kelas?

In [10]:
tanpa_stratify = sum(
    len(set(train_test_split(X, y, test_size=0.3, random_state=rs)[3])) < len(CLASSES)
    for rs in range(100)
)
dengan_stratify = sum(
    len(set(train_test_split(X, y, test_size=0.3, random_state=rs, stratify=y)[3])) < len(CLASSES)
    for rs in range(100)
)

print(f'Dari 100 split tanpa stratify : {tanpa_stratify} split kehilangan minimal 1 kelas di test set')
print(f'Dari 100 split dengan stratify: {dengan_stratify} split kehilangan minimal 1 kelas di test set')

Dari 100 split tanpa stratify : 71 split kehilangan minimal 1 kelas di test set
Dari 100 split dengan stratify: 0 split kehilangan minimal 1 kelas di test set


## Ringkasan

| Parameter | Kalau tidak dipakai | Kalau dipakai |
|---|---|---|
| `random_state` | Split berbeda setiap run, hasil tidak bisa direproduksi | Split terkunci, eksperimen bisa diulang dan dibandingkan |
| `stratify=y` | Proporsi kelas bisa melenceng, bahkan ada kelas yang hilang dari test set | Semua kelas tetap terwakili dengan proporsi mengikuti data asli |

Praktik yang disarankan untuk kasus klasifikasi (biner maupun multi-kelas):

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```

## Export tabel ke PNG untuk slide

Jalankan sel di bawah kalau tabelnya mau dipakai sebagai gambar di PowerPoint.

In [11]:
import dataframe_image as dfi

dfi.export(df, 'split_data_original.png', table_conversion='matplotlib', dpi=200)
dfi.export(distribusi, 'split_class_distribution.png', table_conversion='matplotlib', dpi=200)
dfi.export(df.loc[X_test_ns.index], 'split_test_no_stratify.png', table_conversion='matplotlib', dpi=200)
dfi.export(df.loc[X_test_s.index], 'split_test_stratify.png', table_conversion='matplotlib', dpi=200)
dfi.export(df.loc[X_train_s.index], 'split_train_stratify.png', table_conversion='matplotlib', dpi=200)
dfi.export(perbandingan, 'split_class_proportions.png', table_conversion='matplotlib', dpi=200)